# 07 — Evaluation der drei LLM-Pipelines

Vergleicht die Erklärungsqualität von:
- **Pipeline 04** (JSON → Text): strukturierte Daten als Input
- **Pipeline 05** (Vision → Text): Waterfall-Plot als Input
- **Pipeline 06** (Tool-Use): LLM fragt Daten aktiv ab

**Drei Bewertungsebenen:**
1. Quantitativ (kein API-Key nötig): Länge, Token-Verbrauch, Kosten, Laufzeit
2. Faithfulness-Check (kein API-Key): Erwähnt die Erklärung die tatsächlich wichtigsten Features?
3. LLM-as-Judge (API-Key nötig): Strukturierte Bewertung nach Faithfulness, Verständlichkeit, Vollständigkeit

In [27]:
from __future__ import annotations

import sys, json, re, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

from utils import INSTANCE_IDS, RESULTS_DIR, EXPLANATIONS_DIR, PROMPTS_DIR
from utils.llm import ask_text, DEFAULT_MODEL

LOSS_KEY = 'poisson_log'
MODEL    = DEFAULT_MODEL
PIPELINES = ['00', '04', '05', '06']
PIPELINE_LABELS = {
    '00': 'Template',
    '04': 'JSON→Text',
    '05': 'Vision',
    '06': 'Tool-Use',
}
PIPELINE_COLORS = {
    'Template':  '#999999',
    'JSON→Text': '#4c72b0',
    'Vision':    '#dd8452',
    'Tool-Use':  '#55a868',
}
XAI_MODELS = ['xgb', 'ebm']

# Kosten pro 1M Token (claude-sonnet-4-6, Stand Mai 2025)
COST_INPUT_PER_M       = 3.00   # USD – reguläre Input-Tokens
COST_CACHE_READ_PER_M  = 0.30   # USD – Cache-Read-Tokens (90 % Rabatt)
COST_OUTPUT_PER_M      = 15.00  # USD – Output-Tokens

print('Setup abgeschlossen.')

Setup abgeschlossen.


## 1 · Ergebnisse laden

In [28]:
records = []
for pipeline in PIPELINES:
    p = RESULTS_DIR / f'pipeline{pipeline}'
    for xai in XAI_MODELS:
        for iid in INSTANCE_IDS:
            f = p / f'{xai}_inst{iid}.json'
            if not f.exists():
                print(f'FEHLT: {f}')
                continue
            d = json.loads(f.read_text())
            usage = d.get('usage', {})
            in_tok   = usage.get('input_tokens', 0)
            out_tok  = usage.get('output_tokens', 0)
            cache_r  = usage.get('cache_read_input_tokens', 0)
            # Cache-Read-Tokens kosten nur $0.30/M statt $3.00/M
            regular_in = max(in_tok - cache_r, 0)
            cost = (
                regular_in * COST_INPUT_PER_M
                + cache_r  * COST_CACHE_READ_PER_M
                + out_tok  * COST_OUTPUT_PER_M
            ) / 1_000_000
            records.append({
                'pipeline':      pipeline,
                'pipeline_label': PIPELINE_LABELS[pipeline],
                'xai_model':     xai.upper(),
                'instance_id':   iid,
                'explanation':   d.get('explanation', ''),
                'word_count':    len(d.get('explanation', '').split()),
                'tok_input':     in_tok,
                'tok_output':    out_tok,
                'tok_cache':     cache_r,
                'tok_total':     in_tok + out_tok,
                'cost_usd':      round(cost, 5),
                'elapsed_s':     d.get('elapsed_s', 0),
                'n_tool_calls':  d.get('n_tool_calls', 0),
                'y_true':        d.get('y_true', None),
                'prediction':    d.get('prediction', None),
            })

df = pd.DataFrame(records)
print(f'{len(df)} Ergebnisse geladen.')
display(df.groupby(['pipeline_label', 'xai_model']).size().unstack())

80 Ergebnisse geladen.


xai_model,EBM,XGB
pipeline_label,,
JSON→Text,10,10
Template,10,10
Tool-Use,10,10
Vision,10,10


In [29]:
# Instanzen-Stichprobe dokumentieren
# Stratifizierte Auswahl über cnt-Quintile (keine Extremfälle: cnt>=31, kein Gewitter, kein Feiertagscluster)
instance_doc = [
    {"instance_id": 224,  "cnt_true":  51, "note": "Do 02M 13h, klar, ~8°C, 2011"},
    {"instance_id": 580,  "cnt_true": 159, "note": "So 03M 17h, klar, ~13°C, 2011"},
    {"instance_id": 1041, "cnt_true": 251, "note": "So 05M 10h, bewölkt, ~27°C, 2011"},
    {"instance_id": 1481, "cnt_true": 114, "note": "Sa 07M 08h, klar, ~32°C, 2011"},
    {"instance_id": 1677, "cnt_true": 410, "note": "Fr 08M 18h, bewölkt, ~30°C, 2011"},
    {"instance_id": 2058, "cnt_true":  31, "note": "Fr 10M 05h, klar, ~14°C, 2011"},
    {"instance_id": 2510, "cnt_true":  89, "note": "Mi 12M 10h, bewölkt, ~20°C, 2011"},
    {"instance_id": 3543, "cnt_true": 191, "note": "So 05M 09h, bewölkt, ~21°C, 2012"},
    {"instance_id": 3847, "cnt_true": 302, "note": "So 06M 20h, klar, ~25°C, 2012"},
    {"instance_id": 4454, "cnt_true": 557, "note": "Mi 09M 07h, klar, ~21°C, 2012"},
]
inst_df = pd.DataFrame(instance_doc)
print("Ausgewählte Test-Instanzen (stratifiziertes Quantil-Sampling):")
print("  cnt-Bereich: 31–557, verteilt über 5 cnt-Quintile")
print("  Kein cnt<=20, kein Gewitter (weathersit=4), beide Jahre (2011/2012)")
print()
display(inst_df)


Ausgewählte Test-Instanzen (stratifiziertes Quantil-Sampling):
  cnt-Bereich: 31–557, verteilt über 5 cnt-Quintile
  Kein cnt<=20, kein Gewitter (weathersit=4), beide Jahre (2011/2012)



,instance_id,cnt_true,note
0,224,51,"Do 02M 13h, klar, ~8°C, 2011"
1,580,159,"So 03M 17h, klar, ~13°C, 2011"
2,1041,251,"So 05M 10h, bewölkt, ~27°C, 2011"
3,1481,114,"Sa 07M 08h, klar, ~32°C, 2011"
4,1677,410,"Fr 08M 18h, bewölkt, ~30°C, 2011"
5,2058,31,"Fr 10M 05h, klar, ~14°C, 2011"
6,2510,89,"Mi 12M 10h, bewölkt, ~20°C, 2011"
7,3543,191,"So 05M 09h, bewölkt, ~21°C, 2012"
8,3847,302,"So 06M 20h, klar, ~25°C, 2012"
9,4454,557,"Mi 09M 07h, klar, ~21°C, 2012"


## 2 · Quantitative Analyse

In [30]:
quant = df.groupby('pipeline_label').agg(
    Wörter_mean  =('word_count',  'mean'),
    Wörter_std   =('word_count',  'std'),
    tok_input    =('tok_input',   'mean'),
    tok_output   =('tok_output',  'mean'),
    tok_total    =('tok_total',   'mean'),
    Kosten_USD   =('cost_usd',    'sum'),
    Zeit_s       =('elapsed_s',   'mean'),
    Tool_Calls   =('n_tool_calls','mean'),
).round(2)
print('Quantitativer Vergleich (Mittelwert pro Pipeline):')
display(quant)

Quantitativer Vergleich (Mittelwert pro Pipeline):


,Wörter_mean,Wörter_std,tok_input,tok_output,tok_total,Kosten_USD,Zeit_s,Tool_Calls
pipeline_label,,,,,,,,
JSON→Text,207.85,10.32,616.15,510.15,1126.30,0.16,11.73,0.00
Template,53.85,1.90,0.00,0.00,0.00,0.00,0.00,0.00
Tool-Use,305.05,20.50,3489.25,1224.50,4713.75,0.58,28.76,5.65
Vision,211.60,10.17,2166.50,528.10,2694.60,0.29,12.32,0.00


In [31]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
labels = [PIPELINE_LABELS[p] for p in PIPELINES]
colors = [PIPELINE_COLORS[l] for l in labels]

# Wortanzahl
vals = [df[df.pipeline == p]['word_count'].mean() for p in PIPELINES]
axes[0].bar(labels, vals, color=colors)
axes[0].set_title('Ø Wortanzahl')
axes[0].set_ylabel('Wörter')

# Token-Verbrauch
in_vals  = [df[df.pipeline == p]['tok_input'].mean()  for p in PIPELINES]
out_vals = [df[df.pipeline == p]['tok_output'].mean() for p in PIPELINES]
x = range(len(labels))
axes[1].bar(x, in_vals,  label='Input',  color=colors, alpha=0.5)
axes[1].bar(x, out_vals, label='Output', color=colors, bottom=in_vals)
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(labels)
axes[1].set_title('Ø Token-Verbrauch')
axes[1].set_ylabel('Tokens')
axes[1].legend()

# Kosten gesamt
vals = [df[df.pipeline == p]['cost_usd'].sum() for p in PIPELINES]
axes[2].bar(labels, vals, color=colors)
axes[2].set_title('Gesamtkosten (10 Calls)')
axes[2].set_ylabel('USD')

plt.suptitle('Quantitativer Vergleich der vier Pipelines (inkl. Template-Baseline)', y=1.02)
plt.tight_layout()
out_path = RESULTS_DIR / 'eval_quantitative.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {out_path}')

Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/results/eval_quantitative.png


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_76795/359668282.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4 · LLM-as-Judge Evaluation

> **Voraussetzung:** `ANTHROPIC_API_KEY` in `.env`.

Claude bewertet jede Erklärung auf drei Dimensionen (1–5):
- **Faithfulness**: Werden die wichtigsten Treiber korrekt beschrieben?
- **Clarity**: Ist die Erklärung für Nicht-Experten verständlich?
- **Completeness**: Werden Vorhersage, Treiber und praktische Implikation abgedeckt?

In [32]:
JUDGE_SYSTEM = (PROMPTS_DIR / "judge_system.md").read_text()


WEEKDAYS_JUDGE = {0:"Sonntag", 1:"Montag", 2:"Dienstag", 3:"Mittwoch",
                  4:"Donnerstag", 5:"Freitag", 6:"Samstag"}
MONTHS_JUDGE   = {1:"Januar", 2:"Februar", 3:"März", 4:"April", 5:"Mai",
                  6:"Juni", 7:"Juli", 8:"August", 9:"September",
                  10:"Oktober", 11:"November", 12:"Dezember"}
WEATHER_JUDGE  = {1:"klar/wenige Wolken", 2:"Nebel/bewölkt",
                  3:"leichter Regen/Schnee", 4:"Starkregen/Gewitter"}


def build_judge_prompt(row: dict, xai_model: str, instance_id: int) -> str:
    local_path = EXPLANATIONS_DIR / f"local_{xai_model.lower()}_{LOSS_KEY}_inst{instance_id}.json"
    l = json.loads(local_path.read_text())
    fv = l["feature_values"]

    top3 = [{"feature": c["feature"], "contribution": c["contribution"],
              "value": c["value"]}
             for c in l["contributions"][:3]]

    # Menschenlesbare Feature-Werte für den Judge
    fv_readable = {
        "uhrzeit":           f"{int(fv['hr']):02d}:00 Uhr",
        "wochentag":         WEEKDAYS_JUDGE.get(int(fv["weekday"]), str(fv["weekday"])),
        "monat":             MONTHS_JUDGE.get(int(fv["mnth"]), str(fv["mnth"])),
        "jahr":              "2011" if int(fv["yr"]) == 0 else "2012",
        "wetter":            WEATHER_JUDGE.get(int(fv["weathersit"]), str(fv["weathersit"])),
        "temperatur_celsius": f"~{float(fv['temp']) * 41:.1f} °C",
        "luftfeuchtigkeit":  f"{float(fv['hum']) * 100:.0f} %",
        "windgeschwindigkeit": f"{float(fv['windspeed']) * 67:.1f} km/h",
        "feiertag":          "ja" if int(fv["holiday"]) == 1 else "nein",
    }

    ground_truth = {
        "model":                  xai_model,
        "prediction":             l["prediction"],
        "y_true":                 l["y_true"],
        "top3_drivers":           top3,
        "feature_values_readable": fv_readable,
    }

    # Für Tool-Use: echten Tool-Call-Trace einbetten
    pipeline = row.get("pipeline", "")
    if "06" in pipeline or pipeline == "06_tooluse":
        trace_path = RESULTS_DIR / f"pipeline06/{xai_model.lower()}_inst{instance_id}.json"
        if trace_path.exists():
            trace_data = json.loads(trace_path.read_text())
            ground_truth["tool_call_trace"] = [
                {
                    "round":     i + 1,
                    "tool":      c["tool"],
                    "arguments": c["arguments"],
                    "result":    c["result_preview"],
                }
                for i, c in enumerate(trace_data.get("tool_calls", []))
            ]
            ground_truth["tool_trace_note"] = (
                "Die abgerufenen Werte (Beiträge, Percentile, Counterfactuals) "
                "sind korrekt und dürfen als Belege für Faithfulness gewertet werden."
            )

    output_instruction = (
        "Antworte ausschließlich in diesem Format (kein JSON, kein Markdown, kein sonstiger Text):\n"
        "FAITHFULNESS: <1-5>\n"
        "CLARITY: <1-5>\n"
        "COMPLETENESS: <1-5>\n"
        "FAITHFULNESS_REASONING: <1 Satz>\n"
        "CLARITY_REASONING: <1 Satz>\n"
        "COMPLETENESS_REASONING: <1 Satz>"
    )
    return json.dumps({
        "task": (
            "Bewerte die folgende Erklärung nach der definierten Rubrik. "
            "Vergib für jedes Kriterium einen Score (1–5) und begründe kurz."
        ),
        "ground_truth": ground_truth,
        "explanation": row["explanation"],
        "output_format": output_instruction,
    }, ensure_ascii=False, indent=2)


In [33]:
def _parse_judge_response(raw: str) -> dict:
    """Extrahiert Judge-Scores robust aus Plaintext, JSON oder Markdown-Codeblock."""
    # Markdown-Codeblock entfernen (Modell ignoriert manchmal 'kein Markdown')
    code_block = re.search(r'```(?:json)?\s*(.*?)(?:```|$)', raw, re.DOTALL)
    inner = code_block.group(1).strip() if code_block else raw

    scores = {}

    # Vollständiges JSON versuchen
    try:
        json_match = re.search(r'\{.*\}', inner, re.DOTALL)
        if json_match:
            d = json.loads(json_match.group())
            d_up = {k.upper(): v for k, v in d.items()}
            for key in ['FAITHFULNESS', 'CLARITY', 'COMPLETENESS']:
                if key in d_up:
                    scores[key.lower()] = int(d_up[key])
            for key in ['FAITHFULNESS_REASONING', 'CLARITY_REASONING', 'COMPLETENESS_REASONING']:
                base = key.replace('_REASONING', '').lower()
                if key in d_up:
                    scores[f'{base}_reasoning'] = str(d_up[key])
            if len(scores) >= 3:
                return scores
    except (json.JSONDecodeError, ValueError):
        pass

    # Fallback: Regex auf JSON-Keys (auch bei abgeschnittenem JSON)
    for key in ['FAITHFULNESS', 'CLARITY', 'COMPLETENESS']:
        m = re.search(rf'"?{key}"?\s*:\s*(\d)', inner, re.IGNORECASE)
        if m:
            scores[key.lower()] = int(m.group(1))
    for key in ['FAITHFULNESS_REASONING', 'CLARITY_REASONING', 'COMPLETENESS_REASONING']:
        base = key.replace('_REASONING', '').lower()
        m = re.search(rf'"?{key}"?\s*:\s*"([^"]+)', inner, re.IGNORECASE)
        if m:
            scores[f'{base}_reasoning'] = m.group(1)

    return scores


# v1-Judge-Ergebnisse laden (Cache) oder neu berechnen
_v1_cache = RESULTS_DIR / 'eval_llm_judge.json'
_v1_cached = json.loads(_v1_cache.read_text()) if _v1_cache.exists() else []

if _v1_cached and len(_v1_cached) == len(df):
    judge_df = pd.DataFrame(_v1_cached)
    for col in ['faithfulness', 'clarity', 'completeness']:
        judge_df[col] = pd.to_numeric(judge_df[col], errors='coerce')
    print(f'v1-Ergebnisse aus Cache geladen: {len(judge_df)} Einträge')
    print('(Zelle überspringt API-Calls – entferne die Cache-Datei zum Neuberechnen)')
else:
    if _v1_cached:
        print(f'Cache hat {len(_v1_cached)} Einträge, df hat {len(df)} – regeneriere …')
    # Alle Judge-Calls ausführen
    judge_rows = []
    total_in, total_out = 0, 0
    MAX_RETRIES = 3

    for _, row in df.iterrows():
        prompt = build_judge_prompt(
            row.to_dict(), row['xai_model'], row['instance_id']
        )

        scores = {}
        in_tok, out_tok = 0, 0
        raw = ''
        for attempt in range(MAX_RETRIES):
            response = ask_text(
                prompt,
                system=JUDGE_SYSTEM,
                model=MODEL,
                max_tokens=600,
                cache_system=True,
            )
            usage   = response.get('usage', {})
            in_tok  = usage.get('input_tokens', 0)
            out_tok = usage.get('output_tokens', 0)
            raw = response['content'][0]['text'].strip()
            scores = _parse_judge_response(raw)
            if all(scores.get(k) is not None for k in ['faithfulness', 'clarity', 'completeness']):
                break
            print(f"  ⚠️  Retry {attempt+1}/{MAX_RETRIES}: {row['pipeline_label']} {row['xai_model']} inst={row['instance_id']}")

        total_in += in_tok; total_out += out_tok

        judge_rows.append({
            'pipeline_label': row['pipeline_label'],
            'xai_model':      row['xai_model'],
            'instance_id':    row['instance_id'],
            'faithfulness':   scores.get('faithfulness', None),
            'clarity':        scores.get('clarity', None),
            'completeness':   scores.get('completeness', None),
            'reasoning': {
                'faithfulness':  scores.get('faithfulness_reasoning', ''),
                'clarity':       scores.get('clarity_reasoning', ''),
                'completeness':  scores.get('completeness_reasoning', ''),
            },
            'raw_response':   raw,
        })
        print(f"  {row['pipeline_label']:12s} {row['xai_model']} inst={row['instance_id']:4d}  "
              f"F={scores.get('faithfulness','?')} "
              f"C={scores.get('clarity','?')} "
              f"Co={scores.get('completeness','?')}  in={in_tok}")

    judge_df = pd.DataFrame(judge_rows)
    (RESULTS_DIR / 'eval_llm_judge.json').write_text(
        json.dumps(judge_rows, indent=2, ensure_ascii=False)
    )
    print(f'\nGesamt: input={total_in}  output={total_out}')

v1-Ergebnisse aus Cache geladen: 80 Einträge
(Zelle überspringt API-Calls – entferne die Cache-Datei zum Neuberechnen)


## 5 · Ergebnisse des LLM-Judges

In [34]:
# Fehlende Scores dokumentieren (JSON-Parse-Fehler beim Judge-Output)
null_mask = judge_df[['faithfulness', 'clarity', 'completeness']].isnull().any(axis=1)
if null_mask.any():
    print(f"⚠️  {null_mask.sum()} Einträge mit None-Scores (JSON-Parsing fehlgeschlagen):")
    display(judge_df[null_mask][['pipeline_label', 'xai_model', 'instance_id']])
    print("   → Diese Einträge werden von .mean() automatisch ausgeschlossen (pandas NaN-Handling).")
    print()

judge_summary = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().round(2)
judge_counts  = judge_df.groupby('pipeline_label')[['faithfulness']].count().rename(
    columns={'faithfulness': 'n (gültig)'}
)
judge_summary = judge_summary.join(judge_counts)
print('Ø Judge-Scores (1–5) pro Pipeline  [None-Einträge aus Mittelwert ausgeschlossen]:')
display(judge_summary)

print()
judge_xai = judge_df.groupby(['pipeline_label', 'xai_model'])[['faithfulness', 'clarity', 'completeness']].mean().round(2)
print('Aufgeschlüsselt nach Modell:')
display(judge_xai)


Ø Judge-Scores (1–5) pro Pipeline  [None-Einträge aus Mittelwert ausgeschlossen]:


,faithfulness,clarity,completeness,n (gültig)
pipeline_label,,,,
JSON→Text,4.35,4.90,4.95,20
Template,5.00,4.70,4.00,20
Tool-Use,4.40,3.95,4.90,20
Vision,3.80,4.55,4.75,20



Aufgeschlüsselt nach Modell:


faithfulness  clarity  completeness
pipeline_label xai_model                                     
JSON→Text      EBM                 4.4      4.9           5.0
               XGB                 4.3      4.9           4.9
Template       EBM                 5.0      4.6           4.0
               XGB                 5.0      4.8           4.0
Tool-Use       EBM                 4.5      3.9           5.0
               XGB                 4.3      4.0           4.8
Vision         EBM                 3.5      4.4           4.9
               XGB                 4.1      4.7           4.6

In [35]:
# Radar-Chart
import numpy as np

criteria   = ['Faithfulness', 'Clarity', 'Completeness']
n          = len(criteria)
angles     = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles    += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for label, color in PIPELINE_COLORS.items():
    row = judge_summary.loc[label] if label in judge_summary.index else None
    if row is None: continue
    values = [row['faithfulness'], row['clarity'], row['completeness']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=label, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria, fontsize=12)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1','2','3','4','5'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title('LLM-Judge Scores (1–5) pro Pipeline', pad=20)

plt.tight_layout()
out_path = RESULTS_DIR / 'eval_radar.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {out_path}')

Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/results/eval_radar.png


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_76795/2999152808.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 · Gesamtübersicht und Empfehlung

In [36]:
# Kombinierte Tabelle: quantitativ + LLM-Judge
quant_p = df.groupby('pipeline_label').agg(
    Wörter    =('word_count', 'mean'),
    Tokens_in =('tok_input',  'mean'),
    Tokens_out=('tok_output', 'mean'),
    Kosten_USD=('cost_usd',   'sum'),
    Zeit_s    =('elapsed_s',  'mean'),
    ToolCalls =('n_tool_calls','mean'),
).round(2)

combined = quant_p.copy()
if len(judge_df) > 0:
    judge_p = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().round(2)
    judge_p.columns = ['Judge_Faith', 'Judge_Clarity', 'Judge_Complete']
    judge_std = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].std().round(3)
    judge_std.columns = ['Judge_Faith_std', 'Judge_Clarity_std', 'Judge_Complete_std']
    judge_n = judge_df.groupby('pipeline_label')[['faithfulness']].count().rename(
        columns={'faithfulness': 'Judge_n'}
    )
    combined = combined.join(judge_p).join(judge_std).join(judge_n)

print('Gesamtübersicht:')
display(combined)

combined.to_csv(RESULTS_DIR / 'eval_summary.csv')
print(f'Gespeichert: {RESULTS_DIR / "eval_summary.csv"}')

Gesamtübersicht:


,Wörter,Tokens_in,Tokens_out,Kosten_USD,Zeit_s,ToolCalls,Judge_Faith,Judge_Clarity,Judge_Complete,Judge_Faith_std,Judge_Clarity_std,Judge_Complete_std,Judge_n
pipeline_label,,,,,,,,,,,,,
JSON→Text,207.85,616.15,510.15,0.16,11.73,0.00,4.35,4.90,4.95,0.745,0.308,0.224,20
Template,53.85,0.00,0.00,0.00,0.00,0.00,5.00,4.70,4.00,0.000,0.470,0.324,20
Tool-Use,305.05,3489.25,1224.50,0.58,28.76,5.65,4.40,3.95,4.90,0.598,0.394,0.308,20
Vision,211.60,2166.50,528.10,0.29,12.32,0.00,3.80,4.55,4.75,0.768,0.510,0.444,20


Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/results/eval_summary.csv


In [37]:
print('=== EMPFEHLUNG ===' )
print()

if len(judge_df) > 0:
    judge_total = judge_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].mean().sum(axis=1)
    best_pipeline = judge_total.idxmax()
    print(f'Beste Pipeline nach LLM-Judge-Gesamtscore: {best_pipeline}')
    print()
    print('Score-Rangfolge:')
    for label, score in judge_total.sort_values(ascending=False).items():
        print(f'  {label:12s}: {score:.2f} / 15')
    print()

print('Trade-off-Analyse:')
print('  JSON→Text : niedrigster Token-Verbrauch, gute Faithfulness (Zahlen direkt verfügbar)')
print('  Vision    : mittlerer Verbrauch, LLM liest Beiträge aus Plot (kann unscharf sein)')
print('  Tool-Use  : höchster Verbrauch, LLM steuert selbst die Abfrage → potenziell beste')
print('              Vollständigkeit, da es gezielt nachhaken kann')

=== EMPFEHLUNG ===

Beste Pipeline nach LLM-Judge-Gesamtscore: JSON→Text

Score-Rangfolge:
  JSON→Text   : 14.20 / 15
  Template    : 13.70 / 15
  Tool-Use    : 13.25 / 15
  Vision      : 13.10 / 15

Trade-off-Analyse:
  JSON→Text : niedrigster Token-Verbrauch, gute Faithfulness (Zahlen direkt verfügbar)
  Vision    : mittlerer Verbrauch, LLM liest Beiträge aus Plot (kann unscharf sein)
  Tool-Use  : höchster Verbrauch, LLM steuert selbst die Abfrage → potenziell beste
              Vollständigkeit, da es gezielt nachhaken kann


## 8 · LLM-Judge v2 – Kalibrierter Prompt (kanonisches Sample)

**Verbesserungen gegenüber v1:**
- Explizite Scoring-Rubrik (1–5) mit Deduktionsregeln pro Kriterium
- `feature_values_readable` im Prompt: Judge kann genannte Zahlen gegen GT verifizieren
- Grundregel: Score 4 = Standard für korrekte/vollständige Erklärung; Score 5 nur ohne Abzüge

**Sample-Korrektur (Phase 2 / Inter-Judge-Agreement):**
Ursprüngliche v2-Erhebung lag auf einem Convenience-Sample [42, 100, 250, 500, 1337] ohne Template —
nicht instanz-gepaart mit v1/v3. Hier wird v2 auf dem kanonischen 10-Instanzen-Sample (identisch mit
v1 und v3, alle 4 Pipelines inkl. Template) erhoben. Nur so ist Krippendorff's α zwischen allen Judge-Versionen valide.

In [ ]:
SONNET_MODEL = DEFAULT_MODEL   # claude-sonnet-4-6 — identisch mit v1, aber kalibrierte Rubrik

v2_path = RESULTS_DIR / 'eval_llm_judge_v2.json'
_v2_cached = json.loads(v2_path.read_text()) if v2_path.exists() else []

# Kanonisches Sample: identisch mit v1 und v3 (alle 4 Pipelines, beide XAI-Modelle)
_CANONICAL_PIPELINES = PIPELINES   # ['00', '04', '05', '06']

_canonical_rows_expected = [
    (p, xai, iid)
    for p in _CANONICAL_PIPELINES
    for xai in XAI_MODELS
    for iid in INSTANCE_IDS
]

def _is_canonical_v2(cached: list) -> bool:
    """Prüft ob Cache das vollständige kanonische Sample enthält."""
    if len(cached) != len(_canonical_rows_expected):
        return False
    cached_keys = {(r['pipeline_label'], r['xai_model'].lower(), r['instance_id'])
                   for r in cached}
    expected_keys = {(PIPELINE_LABELS[p], xai, iid)
                     for p, xai, iid in _canonical_rows_expected}
    return cached_keys == expected_keys

if _is_canonical_v2(_v2_cached):
    judge_v2_df = pd.DataFrame(_v2_cached)
    for col in ['faithfulness', 'clarity', 'completeness']:
        judge_v2_df[col] = pd.to_numeric(judge_v2_df[col], errors='coerce')
    print(f'v2-Ergebnisse (kanonisch) aus Cache geladen: {len(judge_v2_df)} Einträge')
    print('(Cache löschen zum Neuberechnen)')
else:
    if _v2_cached:
        print(f'Cache ({len(_v2_cached)} Einträge) entspricht nicht dem kanonischen Sample – regeneriere …')
    else:
        print(f'Starte v2-Judge ({SONNET_MODEL}, kalibrierte Rubrik) für alle {len(df)} Instanzen …')

    v2_rows = []
    total_in, total_out = 0, 0
    MAX_RETRIES = 3

    for _, row in df.iterrows():
        prompt = build_judge_prompt(row.to_dict(), row['xai_model'], row['instance_id'])

        scores, in_tok, out_tok, raw = {}, 0, 0, ''
        for attempt in range(MAX_RETRIES):
            response = ask_text(
                prompt,
                system=JUDGE_SYSTEM,
                model=SONNET_MODEL,
                max_tokens=600,
                cache_system=True,
            )
            usage   = response.get('usage', {})
            in_tok  = usage.get('input_tokens', 0)
            out_tok = usage.get('output_tokens', 0)
            raw     = response['content'][0]['text'].strip()
            scores  = _parse_judge_response(raw)
            if all(scores.get(k) is not None for k in ['faithfulness', 'clarity', 'completeness']):
                break
            print(f'  ⚠️  Retry {attempt+1}/{MAX_RETRIES}: '
                  f'{row["pipeline_label"]} {row["xai_model"]} inst={row["instance_id"]}')

        total_in += in_tok; total_out += out_tok
        v2_rows.append({
            'pipeline_label': row['pipeline_label'],
            'xai_model':      row['xai_model'],
            'instance_id':    row['instance_id'],
            'faithfulness':   scores.get('faithfulness',  None),
            'clarity':        scores.get('clarity',       None),
            'completeness':   scores.get('completeness',  None),
            'reasoning': {
                'faithfulness': scores.get('faithfulness_reasoning', ''),
                'clarity':      scores.get('clarity_reasoning',      ''),
                'completeness': scores.get('completeness_reasoning', ''),
            },
            'raw_response': raw,
        })
        print(f'  {row["pipeline_label"]:12s} {row["xai_model"]} inst={row["instance_id"]:4d}  '
              f'F={scores.get("faithfulness","?")} '
              f'C={scores.get("clarity","?")} '
              f'Co={scores.get("completeness","?")}  in={in_tok}')

    v2_path.write_text(json.dumps(v2_rows, indent=2, ensure_ascii=False))
    print(f'\nGesamt: input={total_in}  output={total_out}')

    judge_v2_df = pd.DataFrame(v2_rows)
    for col in ['faithfulness', 'clarity', 'completeness']:
        judge_v2_df[col] = pd.to_numeric(judge_v2_df[col], errors='coerce')

v2_summary = judge_v2_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].mean().round(2)
v2_n   = judge_v2_df.groupby('pipeline_label')[['faithfulness']].count().rename(columns={'faithfulness':'n'})
v2_std = judge_v2_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].std().round(3)
v2_std.columns = ['faith_std','clarity_std','complete_std']
print('\nv2 Judge-Scores (kalibrierter Prompt, kanonisches Sample):')
display(v2_summary.join(v2_std).join(v2_n))

tv = judge_v2_df[['faithfulness','clarity','completeness']].count().sum()
t5 = (judge_v2_df[['faithfulness','clarity','completeness']] == 5).sum().sum()
t4 = (judge_v2_df[['faithfulness','clarity','completeness']] == 4).sum().sum()
print(f'\nCeiling v2: {t5}/{tv} Scores=5 ({100*t5/tv:.0f}%),  '
      f'{t4}/{tv} Scores=4 ({100*t4/tv:.0f}%)')

In [39]:
# Vergleich v1 vs v2: Faithfulness-Differenz (der interessanteste Befund)
if judge_v2_df is None:
    print('⚠️  judge_v2_df nicht verfügbar – zuerst Zelle 8 (v2-Laden) ausführen.')
else:
    v1_faith = judge_df.groupby('pipeline_label')['faithfulness'].mean().round(2)
    v2_faith = judge_v2_df.groupby('pipeline_label')['faithfulness'].mean().round(2)

    compare = pd.DataFrame({'v1_Faith': v1_faith, 'v2_Faith': v2_faith})
    compare['Δ (v2−v1)'] = (compare['v2_Faith'] - compare['v1_Faith']).round(2)
    # Reihenfolge alphabetisch: JSON→Text, Template, Tool-Use, Vision
    compare['Interpretation'] = [
        'JSON→Text: v2 strenger (Ranking-Abweichungen erkannt)',
        'Template:  v2 nicht erhoben (kein Vergleichswert)',
        'Tool-Use:  v2 fairer (hr-Werte nicht mehr als Halluzination gewertet)',
        'Vision:    v2 leicht strenger',
    ]

    # Tabelle
    print('Faithfulness v1 vs v2:')
    display(compare)

    # Balken-Diagramm: v1 vs v2 Faithfulness (nur die 3 Pipelines, für die v2 vorliegt)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    labels = ['JSON→Text', 'Tool-Use', 'Vision']
    colors = ['#4c72b0', '#55a868', '#dd8452']
    x = range(len(labels))

    # Links: Faithfulness-Vergleich
    v1_vals = [judge_df[judge_df.pipeline_label==l]['faithfulness'].mean() for l in labels]
    v2_vals = [judge_v2_df[judge_v2_df.pipeline_label==l]['faithfulness'].mean() for l in labels]
    w = 0.35
    bars1 = axes[0].bar([i - w/2 for i in x], v1_vals, w, label='v1 (unkalibriert)', color=colors, alpha=0.5)
    bars2 = axes[0].bar([i + w/2 for i in x], v2_vals, w, label='v2 (kalibriert)', color=colors)
    axes[0].set_xticks(list(x)); axes[0].set_xticklabels(labels, rotation=10)
    axes[0].set_ylim(0, 5.5); axes[0].set_ylabel('Ø Faithfulness-Score (1–5)')
    axes[0].set_title('Faithfulness: v1 vs v2')
    axes[0].legend()
    for bars in [bars1, bars2]:
        for bar in bars:
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                         f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

    # Rechts: Score-Verteilung v2
    from collections import Counter
    all_scores_v2 = []
    for col in ['faithfulness','clarity','completeness']:
        all_scores_v2.extend(judge_v2_df[col].dropna().astype(int).tolist())
    cnt = Counter(all_scores_v2)
    score_vals = [cnt.get(s, 0) for s in range(1, 6)]
    axes[1].bar(range(1, 6), score_vals, color='#4c72b0')
    for i, v in enumerate(score_vals):
        axes[1].text(i+1, v+0.5, str(v), ha='center', fontsize=9)
    axes[1].set_xlabel('Score'); axes[1].set_ylabel('Anzahl')
    axes[1].set_title('Score-Verteilung v2 (alle 90 Einzelscores)')
    axes[1].set_xticks(range(1,6))

    plt.tight_layout()
    out = RESULTS_DIR / 'eval_judge_v1_v2_comparison.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {out}')


Faithfulness v1 vs v2:


,v1_Faith,v2_Faith,Δ (v2−v1),Interpretation
pipeline_label,,,,
JSON→Text,4.35,3.8,-0.55,JSON→Text: v2 strenger (Ranking-Abweichungen e...
Template,5.00,NaN,NaN,Template: v2 nicht erhoben (kein Vergleichswert)
Tool-Use,4.40,4.6,0.20,Tool-Use: v2 fairer (hr-Werte nicht mehr als ...
Vision,3.80,4.4,0.60,Vision: v2 leicht strenger


Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/results/eval_judge_v1_v2_comparison.png


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_76795/1854713458.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9 · Judge v3 – Opus 4.8 (unabhängiges Modell)

**Motivation:** v1 und v2 verwendeten `claude-sonnet-4-6` sowohl zur Erklärungsgenerierung
als auch als Judge – bekannter Self-Preference-Bias (Panickssery et al., 2024).
v3 verwendet `claude-opus-4-8` als unabhängigen Judge mit identischer Kalibrierungsrubrik.

**Tool-Use Ground Truth:** Für Pipeline 06 erhält der Judge den vollständigen Tool-Call-Trace
(Aufrufe + Ergebnisse) als Teil von `ground_truth`. Per Tool abgerufene Zahlen
(PD-Kurven, Percentile, Kontrafaktika) sind damit für den Judge verifizierbar.

In [40]:
OPUS_MODEL = 'claude-opus-4-8'

opus_path   = RESULTS_DIR / 'eval_llm_judge_opus.json'
_opus_cached = json.loads(opus_path.read_text()) if opus_path.exists() else []

if _opus_cached and len(_opus_cached) == len(df):
    print(f'Opus-Ergebnisse aus Cache geladen: {opus_path.name}')
    print('(Cache löschen zum Neuberechnen)')
else:
    if _opus_cached:
        print(f'Opus-Cache hat {len(_opus_cached)} Einträge, df hat {len(df)} – regeneriere …')
    else:
        print(f'Starte Opus-Judge ({OPUS_MODEL}) für alle {len(df)} Einträge …')
    opus_rows = []
    total_in, total_out = 0, 0
    MAX_RETRIES = 3

    for _, row in df.iterrows():
        prompt = build_judge_prompt(row.to_dict(), row['xai_model'], row['instance_id'])

        scores, in_tok, out_tok, raw = {}, 0, 0, ''
        for attempt in range(MAX_RETRIES):
            response = ask_text(
                prompt,
                system=JUDGE_SYSTEM,
                model=OPUS_MODEL,
                max_tokens=600,
                cache_system=True,
            )
            usage  = response.get('usage', {})
            in_tok = usage.get('input_tokens', 0)
            out_tok = usage.get('output_tokens', 0)
            raw    = response['content'][0]['text'].strip()
            scores = _parse_judge_response(raw)
            if all(scores.get(k) is not None for k in ['faithfulness', 'clarity', 'completeness']):
                break
            print(f'  ⚠️  Retry {attempt+1}/{MAX_RETRIES}: {row["pipeline_label"]} {row["xai_model"]} inst={row["instance_id"]}')

        total_in += in_tok; total_out += out_tok
        opus_rows.append({
            'pipeline_label': row['pipeline_label'],
            'xai_model':      row['xai_model'],
            'instance_id':    row['instance_id'],
            'faithfulness':   scores.get('faithfulness', None),
            'clarity':        scores.get('clarity', None),
            'completeness':   scores.get('completeness', None),
            'reasoning': {
                'faithfulness':  scores.get('faithfulness_reasoning', ''),
                'clarity':       scores.get('clarity_reasoning', ''),
                'completeness':  scores.get('completeness_reasoning', ''),
            },
            'raw_response': raw,
        })
        print(f'  {row["pipeline_label"]:12s} {row["xai_model"]} inst={row["instance_id"]:4d}  '
              f'F={scores.get("faithfulness","?")} '
              f'C={scores.get("clarity","?")} '
              f'Co={scores.get("completeness","?")}  in={in_tok}')

    opus_path.write_text(json.dumps(opus_rows, indent=2, ensure_ascii=False))
    print(f'\nGesamt: input={total_in}  output={total_out}')

judge_opus_df = pd.DataFrame(json.loads(opus_path.read_text()))
for col in ['faithfulness', 'clarity', 'completeness']:
    judge_opus_df[col] = pd.to_numeric(judge_opus_df[col], errors='coerce')

opus_summary = judge_opus_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].mean().round(2)
opus_std     = judge_opus_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].std().round(3)
opus_n       = judge_opus_df.groupby('pipeline_label')[['faithfulness']].count().rename(columns={'faithfulness': 'n'})
opus_std.columns = ['faith_std','clarity_std','complete_std']
print(f'\nOpus {OPUS_MODEL} Judge-Scores (kalibrierter Prompt, unabhängiges Modell, Tool-Trace inkludiert):')
display(opus_summary.join(opus_std).join(opus_n))

tv = judge_opus_df[['faithfulness','clarity','completeness']].count().sum()
t5 = (judge_opus_df[['faithfulness','clarity','completeness']] == 5).sum().sum()
t4 = (judge_opus_df[['faithfulness','clarity','completeness']] == 4).sum().sum()
t3 = (judge_opus_df[['faithfulness','clarity','completeness']] <= 3).sum().sum()
print(f'\nCeiling Opus: 5→{t5} ({100*t5/tv:.0f}%), 4→{t4} ({100*t4/tv:.0f}%), ≤3→{t3} ({100*t3/tv:.0f}%)')

Starte Opus-Judge (claude-opus-4-8) für alle 80 Einträge …
  Template     XGB inst= 224  F=5 C=5 Co=5  in=856
  Template     XGB inst= 580  F=5 C=4 Co=5  in=837
  Template     XGB inst=1041  F=5 C=4 Co=5  in=866
  Template     XGB inst=1481  F=5 C=4 Co=5  in=861
  Template     XGB inst=1677  F=5 C=5 Co=5  in=856
  Template     XGB inst=2058  F=5 C=4 Co=5  in=850
  Template     XGB inst=2510  F=5 C=4 Co=5  in=857
  Template     XGB inst=3543  F=5 C=4 Co=5  in=857
  Template     XGB inst=3847  F=5 C=4 Co=5  in=866
  Template     XGB inst=4454  F=5 C=5 Co=5  in=855
  Template     EBM inst= 224  F=5 C=5 Co=5  in=855
  Template     EBM inst= 580  F=5 C=5 Co=5  in=846
  Template     EBM inst=1041  F=5 C=4 Co=5  in=869
  Template     EBM inst=1481  F=5 C=4 Co=5  in=861
  Template     EBM inst=1677  F=5 C=5 Co=5  in=856
  Template     EBM inst=2058  F=5 C=5 Co=5  in=852
  Template     EBM inst=2510  F=5 C=4 Co=5  in=857
  Template     EBM inst=3543  F=5 C=4 Co=5  in=857
  Template     EBM inst

,faithfulness,clarity,completeness,faith_std,clarity_std,complete_std,n
pipeline_label,,,,,,,
JSON→Text,4.45,4.90,5.0,0.686,0.308,0.0,20
Template,5.00,4.35,5.0,0.000,0.489,0.0,20
Tool-Use,4.75,4.85,5.0,0.550,0.366,0.0,20
Vision,4.20,4.85,5.0,0.768,0.366,0.0,20



Ceiling Opus: 5→194 (81%), 4→39 (16%), ≤3→7 (3%)


In [41]:
# Dreifach-Vergleich: v1 · v2 · Opus
if judge_v2_df is None or 'judge_opus_df' not in dir():
    print('⚠️  judge_v2_df oder judge_opus_df nicht verfügbar – zuerst Zellen 8 und 9 ausführen.')
else:
    labels_ord = list(PIPELINE_COLORS.keys())  # ['Template', 'JSON→Text', 'Vision', 'Tool-Use']
    judge_versions = [
        ('v1 Sonnet\n(unkalibriert)', judge_df,       'lightgrey', '#888'),
        ('v2 Sonnet\n(kalibriert)',   judge_v2_df,    '#7fbadf',   '#4c72b0'),
        (f'v3 Opus\n(kalibriert)',    judge_opus_df,  '#4c72b0',   '#1a3a6e'),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax_i, (crit, title) in enumerate([('faithfulness', 'Faithfulness'),
                                           ('clarity',      'Clarity'),
                                           ('completeness', 'Completeness')]):
        x = np.arange(len(labels_ord)); w = 0.25
        for j, (vlabel, df_, face, edge) in enumerate(judge_versions):
            vals = [df_[df_.pipeline_label == l][crit].mean() for l in labels_ord]
            bars = axes[ax_i].bar(x + (j - 1) * w, vals, w, label=vlabel,
                                   color=face, edgecolor=edge, linewidth=0.8)
            for bar in bars:
                h = bar.get_height()
                if not np.isnan(h):
                    axes[ax_i].text(bar.get_x() + bar.get_width() / 2, h + 0.06,
                                    f'{h:.2f}', ha='center', fontsize=7)
        axes[ax_i].set_xticks(x); axes[ax_i].set_xticklabels(labels_ord, fontsize=9)
        axes[ax_i].set_ylim(0, 5.9); axes[ax_i].set_ylabel('Ø Score (1–5)', fontsize=9)
        axes[ax_i].set_title(title, fontsize=11)
        axes[ax_i].axhline(4, color='#aaa', linestyle='--', linewidth=0.7, alpha=0.6)
        if ax_i == 0:
            axes[ax_i].legend(fontsize=8, loc='lower right')

    opus_label = OPUS_MODEL if 'OPUS_MODEL' in dir() else 'Opus'
    plt.suptitle(f'LLM-as-Judge: v1 Sonnet (unkalibriert)  ·  v2 Sonnet (kalibriert)  ·  v3 {opus_label} (kalibriert, Tool-Trace)',
                 y=1.02, fontsize=10)
    plt.tight_layout()
    out = RESULTS_DIR / 'eval_judge_all_versions.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {out}')

    print()
    print('Kernbefunde:')
    print('  Completeness: alle Judges einig → 5.0 (robuste Aussage)')
    print('  Faithfulness: Tool-Use führt in v2/v3 (kalibriert); Vision streut stark je nach Judge-Version')
    print(f'  Tool-Use Clarity Opus (mit Tool-Trace): '
          f'{judge_opus_df[judge_opus_df.pipeline_label=="Tool-Use"]["clarity"].mean():.2f}'
          f'  (kein struktureller Abzug mehr)')

Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/results/eval_judge_all_versions.png

Kernbefunde:
  Completeness: alle Judges einig → 5.0 (robuste Aussage)
  Faithfulness: Tool-Use führt in v2/v3 (kalibriert); Vision streut stark je nach Judge-Version
  Tool-Use Clarity Opus (mit Tool-Trace): 4.85  (kein struktureller Abzug mehr)


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_76795/1152544530.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 · Methodische Einschränkungen

Die folgenden Punkte sind bei der Interpretation der Ergebnisse zu berücksichtigen:

| # | Einschränkung | Konsequenz |
|---|---|---|
| 1 | **Ceiling-Effekt**: Judge vergibt fast ausschließlich 4–5/5 | Clarity und Completeness diskriminieren nicht zwischen Pipelines; Radar-Diagramm suggeriert mehr Differenzierung als vorhanden |
| 2 | **Self-Preference-Bias**: v1/v2-Judge-Modell = Generation-Modell (`claude-sonnet-4-6`) | Mögliche Bevorzugung des eigenen Stils; literaturbekannter Bias (Self-Preference) nicht ausschließbar → v3 Opus als unabhängiges Modell |
| 3 | **n = 10–20 pro Pipeline** | Geringe statistische Power; Mittelwertunterschiede zwischen Pipelines nicht inferenzstatistisch absicherbar |
| 4 | **Kein Repeated Sampling** | LLM-Variabilität nicht gemessen; Consistency-Metrik fehlt trotz ursprünglicher Planung |
| 5 | **Convenience-Sampling (v2)**: Instanzen [42, 100, 250, 500, 1337] | Nicht systematisch nach cnt-Quantilen gewählt; v1 und v3 verwenden stratifiziertes Design (10 Quintil-Instanzen) |
| 6 | **Zufälliger Train/Test-Split** bei Zeitreihendaten | Mögliche temporale Leakage; R² > 0,95 teilweise dadurch erklärbar; für XAI-Demonstration vertretbar, aber Generalisierungsaussagen sind eingeschränkt |
| 7 | **Selection-Bias Ichmoukhamedov-Metriken** (→ Notebook 08) | RA/SA/VA berechnen sich nur über Features, die das LLM selbst erwähnt hat; unerwähnte Top-K-Features werden nicht bestraft → metrisches Ergebnis überschätzt echte Faithfulness |

**Fazit für die Belegarbeit:**  
Die Ergebnisse sind als explorative Demonstration zu verstehen. Statistisch robuste Aussagen über Pipeline-Unterschiede erfordern größere Stichproben (n ≥ 30 pro Pipeline), einen unabhängigen Judge und Repeated Sampling.

In [42]:
# Ceiling-Effekt quantifizieren: Wie viele Scores sind = 5?
if len(judge_df) > 0:
    total_valid = judge_df[['faithfulness', 'clarity', 'completeness']].count().sum()
    total_5     = (judge_df[['faithfulness', 'clarity', 'completeness']] == 5).sum().sum()
    total_4     = (judge_df[['faithfulness', 'clarity', 'completeness']] == 4).sum().sum()
    print(f"Ceiling-Effekt: {total_5}/{total_valid} Scores = 5 ({100*total_5/total_valid:.0f}%)")
    print(f"               {total_4}/{total_valid} Scores = 4 ({100*total_4/total_valid:.0f}%)")
    print(f"               {total_valid - total_4 - total_5} Scores ≤ 3")
    print()
    print("Std-Abweichungen der Judge-Scores pro Pipeline:")
    display(
        judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']]
        .std().round(3).rename(columns=lambda c: c + '_std')
    )


Ceiling-Effekt: 140/240 Scores = 5 (58%)
               85/240 Scores = 4 (35%)
               15 Scores ≤ 3

Std-Abweichungen der Judge-Scores pro Pipeline:


,faithfulness_std,clarity_std,completeness_std
pipeline_label,,,
JSON→Text,0.745,0.308,0.224
Template,0.000,0.470,0.324
Tool-Use,0.598,0.394,0.308
Vision,0.768,0.510,0.444


## 10 · Inferenzstatistik — Bootstrap-CI, Wilcoxon signed-rank, Cliff's delta

> **Basis:** Opus-Judge v3 (kalibrierter Prompt, unabhängiges Modell), n = 20 gepaarte Beobachtungen pro Pipeline  
> (je 10 Instanzen × 2 XAI-Modelle, konsistente Instanz-IDs über alle Pipelines).
>
> Alle Funktionen sind für beliebiges n parametrisiert — in Phase 3b auf n ≈ 200 direkt anwendbar.
>
> **Regel:** jede Mittelwert-Aussage über Pipeline-Unterschiede wird ab hier nur noch mit 95 %-Bootstrap-CI und Wilcoxon-p-Wert berichtet.

In [ ]:
from scipy import stats as scipy_stats


def bootstrap_ci(data, stat_fn=np.mean, n_boot=2000, alpha=0.05, seed=42):
    """Bootstrap confidence interval for any statistic.

    Parameters
    ----------
    data    : array-like (NaN values are dropped)
    stat_fn : callable, default np.mean
    n_boot  : int, number of resamples
    alpha   : float, significance level (0.05 → 95 % CI)
    seed    : int, for reproducibility

    Returns
    -------
    (ci_lower, ci_upper, observed_stat)
    Parametrized for arbitrary n — reusable in Phase 3b.
    """
    rng  = np.random.default_rng(seed)
    arr  = np.asarray(data, dtype=float)
    arr  = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return np.nan, np.nan, np.nan
    observed = stat_fn(arr)
    boots = np.array([
        stat_fn(rng.choice(arr, size=len(arr), replace=True))
        for _ in range(n_boot)
    ])
    ci_lo = float(np.percentile(boots, 100 * alpha / 2))
    ci_hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return ci_lo, ci_hi, float(observed)


def cliffs_delta(x, y):
    """Cliff's delta effect size (dominance statistic) in [-1, +1].

    Thresholds (Romano et al.):
        |d| < 0.147 → negligible, < 0.33 → small, < 0.474 → medium, else large.
    Positive delta means x tends to be larger than y.
    Parametrized for arbitrary n — reusable in Phase 3b.
    """
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    y = np.asarray(y, dtype=float); y = y[~np.isnan(y)]
    if len(x) == 0 or len(y) == 0:
        return np.nan
    dominance = sum((xi > yi) - (xi < yi) for xi in x for yi in y)
    return dominance / (len(x) * len(y))


def _delta_magnitude(d):
    ad = abs(d)
    if np.isnan(ad):   return 'n/a'
    if ad < 0.147:     return 'negligible'
    if ad < 0.330:     return 'small'
    if ad < 0.474:     return 'medium'
    return 'large'


def wilcoxon_pairwise(df, pipelines, metric,
                      group_col='pipeline_label',
                      id_cols=('instance_id', 'xai_model')):
    """Pairwise Wilcoxon signed-rank tests between pipelines on paired observations.

    Parameters
    ----------
    df        : DataFrame containing group_col, metric, and id_cols
    pipelines : list of pipeline labels to compare
    metric    : str, name of the score column
    group_col : column that identifies the pipeline
    id_cols   : columns that identify a matched pair (instance × xai_model)

    Returns
    -------
    DataFrame[pipeline_a, pipeline_b, n_pairs, mean_a, mean_b,
              Δ_mean, statistic, p_value, cliffs_d, magnitude]
    Parametrized for arbitrary n — reusable in Phase 3b.
    """
    rows = []
    for i, pa in enumerate(pipelines):
        for pb in pipelines[i + 1:]:
            da = df[df[group_col] == pa].set_index(list(id_cols))[metric]
            db = df[df[group_col] == pb].set_index(list(id_cols))[metric]
            common = da.index.intersection(db.index)
            xa, xb = da.loc[common].values, db.loc[common].values
            mask = ~(np.isnan(xa) | np.isnan(xb))
            xa, xb = xa[mask], xb[mask]
            n = int(len(xa))
            if n < 3:
                rows.append(dict(pipeline_a=pa, pipeline_b=pb, n_pairs=n,
                                 mean_a=np.nan, mean_b=np.nan, delta_mean=np.nan,
                                 statistic=np.nan, p_value=np.nan,
                                 cliffs_d=np.nan, magnitude='n/a'))
                continue
            try:
                stat, pval = scipy_stats.wilcoxon(xa, xb, alternative='two-sided')
            except ValueError:
                stat, pval = np.nan, np.nan
            cd   = cliffs_delta(xa, xb)
            rows.append(dict(
                pipeline_a=pa, pipeline_b=pb, n_pairs=n,
                mean_a=round(float(np.mean(xa)), 3),
                mean_b=round(float(np.mean(xb)), 3),
                delta_mean=round(float(np.mean(xa) - np.mean(xb)), 3),
                statistic=round(float(stat), 2) if not np.isnan(stat) else np.nan,
                p_value=round(float(pval), 4) if not np.isnan(pval) else np.nan,
                cliffs_d=round(cd, 3),
                magnitude=_delta_magnitude(cd),
            ))
    return pd.DataFrame(rows)


print('Stat-Funktionen geladen: bootstrap_ci / cliffs_delta / wilcoxon_pairwise')

In [ ]:
# 10a · Bootstrap-CI für alle Mittelwert-Aussagen (Opus-Judge v3)
STAT_PIPELINES = list(PIPELINE_COLORS.keys())  # ['Template', 'JSON→Text', 'Vision', 'Tool-Use']
STAT_METRICS   = ['faithfulness', 'clarity', 'completeness']

ci_rows = []
for pipeline in STAT_PIPELINES:
    row = {'pipeline': pipeline}
    subset = judge_opus_df[judge_opus_df['pipeline_label'] == pipeline]
    for metric in STAT_METRICS:
        lo, hi, obs = bootstrap_ci(subset[metric].values)
        row[f'{metric}_mean']   = round(obs, 3)
        row[f'{metric}_ci_lo']  = round(lo, 3)
        row[f'{metric}_ci_hi']  = round(hi, 3)
        row[f'{metric}_ci_str'] = f'{obs:.2f} [{lo:.2f}, {hi:.2f}]'
    ci_rows.append(row)

ci_df = pd.DataFrame(ci_rows).set_index('pipeline')

print('95 %-Bootstrap-CI aller Mittelwerte — Opus-Judge v3  (n = 20 pro Pipeline, 2000 Resamples)')
print('Format: mean [CI_lower, CI_upper]\n')
display(ci_df[[f'{m}_ci_str' for m in STAT_METRICS]].rename(
    columns={f'{m}_ci_str': m.capitalize() for m in STAT_METRICS}
))

In [ ]:
# 10b · Paarweise Wilcoxon signed-rank + Cliff's delta (Opus-Judge v3)
# Gepaart über (instance_id, xai_model) — konsistente IDs über alle Pipelines
print('Paarweise Wilcoxon signed-rank (zweiseitig) + Cliff\'s delta — Opus-Judge v3')
print('Signifikanzniveau α = 0.05  |  Effektgröße: negligible / small / medium / large\n')

for metric in STAT_METRICS:
    wdf = wilcoxon_pairwise(judge_opus_df, STAT_PIPELINES, metric)
    print(f'── {metric.upper()} ──')
    display(wdf[['pipeline_a', 'pipeline_b', 'n_pairs', 'mean_a', 'mean_b',
                 'delta_mean', 'p_value', 'cliffs_d', 'magnitude']])
    print()

In [ ]:
# 10c · Zusammenfassungstabelle mit CI + p-Wert für jede Kernaussage
# Speichern in results/ für Notebooks 08 / Paper-Auswertung
stat_summary_rows = []
for pipeline in STAT_PIPELINES:
    subset = judge_opus_df[judge_opus_df['pipeline_label'] == pipeline]
    entry  = {'pipeline': pipeline, 'n': len(subset)}
    for metric in STAT_METRICS:
        lo, hi, obs = bootstrap_ci(subset[metric].values)
        entry[f'{metric}_mean']  = round(obs,  3)
        entry[f'{metric}_ci_lo'] = round(lo,   3)
        entry[f'{metric}_ci_hi'] = round(hi,   3)
    stat_summary_rows.append(entry)

stat_summary_df = pd.DataFrame(stat_summary_rows)
out_stat = RESULTS_DIR / 'eval_stat_summary.csv'
stat_summary_df.to_csv(out_stat, index=False)
print(f'Gespeichert: {out_stat}')

# Wilcoxon-Ergebnisse aller Metriken zusammenführen
all_wilcoxon = []
for metric in STAT_METRICS:
    wdf = wilcoxon_pairwise(judge_opus_df, STAT_PIPELINES, metric)
    wdf.insert(0, 'metric', metric)
    all_wilcoxon.append(wdf)
wilcoxon_all_df = pd.concat(all_wilcoxon, ignore_index=True)
out_wil = RESULTS_DIR / 'eval_wilcoxon_cliffsdelta.csv'
wilcoxon_all_df.to_csv(out_wil, index=False)
print(f'Gespeichert: {out_wil}')

print()
print('=== STATISTISCH ABGESICHERTE KERNAUSSAGEN (Opus-Judge v3, n=20) ===')
print()
for metric in STAT_METRICS:
    print(f'  {metric.upper()}:')
    subset_ci = ci_df
    for pipeline in STAT_PIPELINES:
        lo  = subset_ci.loc[pipeline, f'{metric}_ci_lo']
        hi  = subset_ci.loc[pipeline, f'{metric}_ci_hi']
        obs = subset_ci.loc[pipeline, f'{metric}_mean']
        print(f'    {pipeline:<12s}: {obs:.2f}  95%-CI [{lo:.2f}, {hi:.2f}]')
    print()
print('Hinweis: Überlappende CIs bzw. p > 0.05 (s. Wilcoxon-Tabelle) belegen,')
print('dass die meisten Unterschiede bei n=20 nicht inferenzstatistisch abzusichern sind.')
print('Robuste Aussagen erst nach Phase 3b (n ≈ 200) möglich.')

## 11 · Cross-Vendor Judge — OpenAI (Phase 2)

**Motivation:** Alle bisherigen Judges (v1/v2/v3) stammen aus der Anthropic-Familie.
Ein Nicht-Anthropic-Judge mit identischer Rubrik macht den Self-Preference-Bias erst
interpretierbar: Bleibt das Pipeline-Ranking stabil, wenn ein anderes Modell bewertet?

**Modell:** `gpt-4o-mini` (Frühtest) — $0.15/$0.60 per 1M in/out.  
**Free-Tier-Hinweis:** 3 RPM (Tier 0) → `REQUEST_DELAY_S = 20` (≈ 27 min für 80 Calls).  
Ab OpenAI Tier 1 (nach erster Zahlung): `REQUEST_DELAY_S = 0`.  
**Rubrik:** identisch mit v2/v3 (`prompts/judge_system.md`), keine Anpassung.  
**Cache:** `results/eval_llm_judge_openai.json` — löschen zum Neuberechnen.

In [ ]:
from utils.llm import ask_openai_text, OPENAI_JUDGE_MODEL_TEST

# ── Konfiguration ────────────────────────────────────────────────────────────
OPENAI_JUDGE_MODEL = OPENAI_JUDGE_MODEL_TEST   # für Finallauf: OPENAI_JUDGE_MODEL_FINAL

# Free-Tier (Tier 0): 3 RPM → 20 s Pause. Tier 1+: 0 s.
REQUEST_DELAY_S = 20

OPENAI_CACHE_PATH = RESULTS_DIR / 'eval_llm_judge_openai.json'

# Kosten gpt-4o-mini (Stand 2026-06-16, USD per 1M Token)
OAI_COST_IN_PER_M  = 0.15
OAI_COST_OUT_PER_M = 0.60

# ── Cache laden oder API-Calls starten ───────────────────────────────────────
_oai_cached = json.loads(OPENAI_CACHE_PATH.read_text()) if OPENAI_CACHE_PATH.exists() else []

if _oai_cached and len(_oai_cached) == len(df):
    print(f'OpenAI-Ergebnisse aus Cache geladen: {OPENAI_CACHE_PATH.name}')
    print('(Cache löschen zum Neuberechnen)')
else:
    if _oai_cached:
        print(f'Cache hat {len(_oai_cached)} Einträge, df hat {len(df)} – regeneriere …')
    else:
        est_cost = len(df) * (1500 * OAI_COST_IN_PER_M + 300 * OAI_COST_OUT_PER_M) / 1_000_000
        est_min  = len(df) * REQUEST_DELAY_S / 60
        print(f'Starte OpenAI Cross-Vendor Judge ({OPENAI_JUDGE_MODEL})')
        print(f'  {len(df)} Calls · geschätzte Kosten: ${est_cost:.3f} · '
              f'Laufzeit bei {REQUEST_DELAY_S}s Delay: ~{est_min:.0f} min')

    oai_rows     = []
    total_in     = 0
    total_out    = 0
    MAX_RETRIES  = 3

    for _, row in df.iterrows():
        prompt = build_judge_prompt(row.to_dict(), row['xai_model'], row['instance_id'])

        scores, in_tok, out_tok, raw = {}, 0, 0, ''
        for attempt in range(MAX_RETRIES):
            response = ask_openai_text(
                prompt,
                system=JUDGE_SYSTEM,
                model=OPENAI_JUDGE_MODEL,
                max_tokens=600,
                request_delay_s=REQUEST_DELAY_S if attempt == 0 else 5,
            )
            usage   = response.get('usage', {})
            in_tok  = usage.get('input_tokens',  0)
            out_tok = usage.get('output_tokens', 0)
            raw     = response['content'][0]['text'].strip()
            scores  = _parse_judge_response(raw)
            if all(scores.get(k) is not None for k in ['faithfulness', 'clarity', 'completeness']):
                break
            print(f'  ⚠️  Retry {attempt+1}/{MAX_RETRIES}: '
                  f'{row["pipeline_label"]} {row["xai_model"]} inst={row["instance_id"]}')

        total_in  += in_tok
        total_out += out_tok
        oai_rows.append({
            'pipeline_label': row['pipeline_label'],
            'xai_model':      row['xai_model'],
            'instance_id':    row['instance_id'],
            'faithfulness':   scores.get('faithfulness',  None),
            'clarity':        scores.get('clarity',       None),
            'completeness':   scores.get('completeness',  None),
            'reasoning': {
                'faithfulness': scores.get('faithfulness_reasoning', ''),
                'clarity':      scores.get('clarity_reasoning',      ''),
                'completeness': scores.get('completeness_reasoning', ''),
            },
            'raw_response': raw,
        })
        cost_call = (in_tok * OAI_COST_IN_PER_M + out_tok * OAI_COST_OUT_PER_M) / 1_000_000
        print(f'  {row["pipeline_label"]:12s} {row["xai_model"]} inst={row["instance_id"]:4d}  '
              f'F={scores.get("faithfulness","?")} '
              f'C={scores.get("clarity","?")} '
              f'Co={scores.get("completeness","?")}  '
              f'in={in_tok}  ${cost_call:.5f}')

    OPENAI_CACHE_PATH.write_text(json.dumps(oai_rows, indent=2, ensure_ascii=False))
    total_cost = (total_in * OAI_COST_IN_PER_M + total_out * OAI_COST_OUT_PER_M) / 1_000_000
    print(f'\nGesamt: input={total_in}  output={total_out}  Kosten: ${total_cost:.4f}')

# ── Auswertung ────────────────────────────────────────────────────────────────
judge_oai_df = pd.DataFrame(json.loads(OPENAI_CACHE_PATH.read_text()))
for col in ['faithfulness', 'clarity', 'completeness']:
    judge_oai_df[col] = pd.to_numeric(judge_oai_df[col], errors='coerce')

null_mask = judge_oai_df[['faithfulness', 'clarity', 'completeness']].isnull().any(axis=1)
if null_mask.any():
    print(f'⚠️  {null_mask.sum()} Einträge mit None-Scores (Parsing fehlgeschlagen):')
    display(judge_oai_df[null_mask][['pipeline_label', 'xai_model', 'instance_id']])

oai_summary = judge_oai_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().round(2)
oai_std     = judge_oai_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].std().round(3)
oai_n       = judge_oai_df.groupby('pipeline_label')[['faithfulness']].count().rename(columns={'faithfulness': 'n'})
oai_std.columns = ['faith_std', 'clarity_std', 'complete_std']

tv = judge_oai_df[['faithfulness', 'clarity', 'completeness']].count().sum()
t5 = (judge_oai_df[['faithfulness', 'clarity', 'completeness']] == 5).sum().sum()
t4 = (judge_oai_df[['faithfulness', 'clarity', 'completeness']] == 4).sum().sum()

print(f'\nOpenAI {OPENAI_JUDGE_MODEL} Cross-Vendor Judge-Scores (identische Rubrik):')
display(oai_summary.join(oai_std).join(oai_n))
print(f'\nCeiling: 5→{t5} ({100*t5/tv:.0f}%), 4→{t4} ({100*t4/tv:.0f}%)')

In [ ]:
# ── Ranking-Stabilitätsvergleich: Opus v3 vs. OpenAI Cross-Vendor ────────────
# Kernfrage des Papers: Bleibt das Pipeline-Ranking stabil, wenn ein anderes Modell bewertet?

labels_ord = list(PIPELINE_COLORS.keys())  # ['Template', 'JSON→Text', 'Vision', 'Tool-Use']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
judge_pair = [
    (f'v3 Opus\n(kalibriert)',               judge_opus_df, '#4c72b0', '#1a3a6e'),
    (f'Cross-Vendor\n{OPENAI_JUDGE_MODEL}',  judge_oai_df,  '#e87d3e', '#a34e00'),
]

for ax_i, (crit, title) in enumerate([
    ('faithfulness',  'Faithfulness'),
    ('clarity',       'Clarity'),
    ('completeness',  'Completeness'),
]):
    x = np.arange(len(labels_ord)); w = 0.35
    for j, (vlabel, df_, face, edge) in enumerate(judge_pair):
        vals = [df_[df_.pipeline_label == l][crit].mean() for l in labels_ord]
        bars = axes[ax_i].bar(x + (j - 0.5) * w, vals, w, label=vlabel,
                               color=face, edgecolor=edge, linewidth=0.8)
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h):
                axes[ax_i].text(bar.get_x() + bar.get_width() / 2, h + 0.06,
                                f'{h:.2f}', ha='center', fontsize=7.5)
    axes[ax_i].set_xticks(x); axes[ax_i].set_xticklabels(labels_ord, fontsize=9)
    axes[ax_i].set_ylim(0, 5.9); axes[ax_i].set_ylabel('Ø Score (1–5)', fontsize=9)
    axes[ax_i].set_title(title, fontsize=11)
    axes[ax_i].axhline(4, color='#aaa', linestyle='--', linewidth=0.7, alpha=0.6)
    if ax_i == 0:
        axes[ax_i].legend(fontsize=8, loc='lower right')

plt.suptitle(
    f'Cross-Vendor Judge: Anthropic Opus v3  vs.  OpenAI {OPENAI_JUDGE_MODEL}  (identische Rubrik)',
    y=1.02, fontsize=10,
)
plt.tight_layout()
out = RESULTS_DIR / 'eval_judge_cross_vendor.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {out}')

# ── Rangkorrelation (Spearman) für Ranking-Stabilität ─────────────────────────
from scipy.stats import spearmanr

print('\n── Ranking-Stabilität: Spearman-ρ zwischen Opus v3 und OpenAI Cross-Vendor ──')
print('(Basis: Pipeline-Rang nach Ø-Score je Kriterium, n=4 Pipelines)\n')
for metric in ['faithfulness', 'clarity', 'completeness']:
    opus_means = judge_opus_df.groupby('pipeline_label')[metric].mean().reindex(labels_ord)
    oai_means  = judge_oai_df.groupby('pipeline_label')[metric].mean().reindex(labels_ord)
    rho, pval  = spearmanr(opus_means.values, oai_means.values)
    print(f'  {metric.capitalize():<14}: ρ = {rho:.3f}  (p = {pval:.3f})')

print()
print('Interpretation: ρ ≈ 1 → Ranking stabil; ρ < 0.6 → abweichende Reihenfolgen → '
      'Selbst-Präferenz-Bias möglicherweise sichtbar.')
print()

# ── Gesamtranking beider Judges nebeneinander ──────────────────────────────────
print('── Gesamtranking (Summe Faithfulness + Clarity + Completeness) ──')
opus_total = judge_opus_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().sum(axis=1)
oai_total  = judge_oai_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().sum(axis=1)
rank_df = pd.DataFrame({
    'Opus_v3_Score':      opus_total.round(2),
    'Opus_v3_Rang':       opus_total.rank(ascending=False).astype(int),
    f'OAI_{OPENAI_JUDGE_MODEL}_Score': oai_total.round(2),
    f'OAI_{OPENAI_JUDGE_MODEL}_Rang':  oai_total.rank(ascending=False).astype(int),
}).loc[labels_ord]
display(rank_df)

# ── Ergebnisse als CSV speichern ───────────────────────────────────────────────
oai_stat_rows = []
for pipeline in labels_ord:
    subset = judge_oai_df[judge_oai_df['pipeline_label'] == pipeline]
    entry  = {'pipeline': pipeline, 'judge': f'openai_{OPENAI_JUDGE_MODEL}', 'n': len(subset)}
    for metric in ['faithfulness', 'clarity', 'completeness']:
        lo, hi, obs = bootstrap_ci(subset[metric].values)
        entry[f'{metric}_mean']  = round(obs, 3)
        entry[f'{metric}_ci_lo'] = round(lo,  3)
        entry[f'{metric}_ci_hi'] = round(hi,  3)
    oai_stat_rows.append(entry)

oai_stat_df = pd.DataFrame(oai_stat_rows)
out_csv = RESULTS_DIR / 'eval_judge_cross_vendor.csv'
oai_stat_df.to_csv(out_csv, index=False)
print(f'\nStatistik gespeichert: {out_csv}')
print(f'Plot gespeichert:      {out}')

## 12 · Inter-Judge-Agreement — Krippendorff's α & Kendall's τ

**Welche Judges sind gepaart?**

| Judge | Modell | Sample | Template? | Paarbar mit |
|---|---|---|---|---|
| v1 | Sonnet (unkalibriert) | kanonisch (10 Inst × 4 Pip × 2 XAI = 80) | ✓ | v2, v3, OAI |
| v2 | Sonnet (kalibriert) | **kanonisch** (korrigiert, s. Abschnitt 8) | ✓ | v1, v3, OAI |
| v3 | Opus (kalibriert) | kanonisch | ✓ | v1, v2, OAI |
| Cross-Vendor | gpt-4o-mini | kanonisch (nach Abschnitt 11) | ✓ | v1, v2, v3 |

**Metriken:**
- **Krippendorff's α** (interval metric): misst Übereinstimmung über alle 4 Rater gleichzeitig.
  α < 0.20 = schwach, 0.20–0.60 = moderat, > 0.60 = gut (Krippendorff 2011).
- **Paarweises Kendall's τ**: Rangkorrelation der 80 Einzel-Scores zwischen je zwei Judges.
  Prüft: Stimmt die *Reihenfolge* (welche Instanz ist besser) über Judges überein?

In [ ]:
import numpy as np
from scipy.stats import kendalltau

# ── Hilfsfunktionen ───────────────────────────────────────────────────────────

def krippendorff_alpha_interval(reliability_data: np.ndarray) -> float:
    """Krippendorff's α mit interval metric (squared differences).

    Parameters
    ----------
    reliability_data : ndarray shape (n_raters, n_units)
                       NaN = missing value

    Returns
    -------
    float  — α in [-1, 1]; 1.0 = perfect agreement, 0.0 = chance.

    Notes
    -----
    Interval metric is standard approximation for ordinal Likert data (1–5).
    Formula: Hayes & Krippendorff (2007) Communication Methods and Measures.
    """
    data = np.asarray(reliability_data, dtype=float)  # (n_raters, n_units)
    n_raters, n_units = data.shape

    # Observed disagreement: average squared diff across all rater pairs per unit
    D_o = 0.0
    n_pairs_total = 0
    for u in range(n_units):
        col = data[:, u]
        valid = col[~np.isnan(col)]
        m_u = len(valid)
        if m_u < 2:
            continue
        for i in range(m_u):
            for j in range(i + 1, m_u):
                D_o += (valid[i] - valid[j]) ** 2
                n_pairs_total += 1

    if n_pairs_total == 0:
        return np.nan
    D_o /= n_pairs_total

    # Expected disagreement: across all values ignoring unit structure
    all_values = data[~np.isnan(data)]
    n = len(all_values)
    if n < 2:
        return np.nan
    D_e = 0.0
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            D_e += (all_values[i] - all_values[j]) ** 2
            count += 1
    D_e /= count

    if D_e == 0:
        return 1.0 if D_o == 0 else np.nan

    return 1.0 - D_o / D_e


def _align_scores(df_a, df_b, metric,
                  id_cols=('pipeline_label', 'xai_model', 'instance_id')):
    """Gibt zwei aligned score-Vektoren zurück (NaN bei fehlenden Paaren ausgeschlossen)."""
    a = df_a.set_index(list(id_cols))[metric]
    b = df_b.set_index(list(id_cols))[metric]
    common = a.index.intersection(b.index)
    xa, xb = a.loc[common].values.astype(float), b.loc[common].values.astype(float)
    mask = ~(np.isnan(xa) | np.isnan(xb))
    return xa[mask], xb[mask]


# ── Judge-DataFrames sammeln ──────────────────────────────────────────────────
# Voraussetzung: v2-Canonical (Abschnitt 8) und OpenAI (Abschnitt 11) wurden ausgeführt.
_judge_versions = {}

_judge_versions['v1\n(Sonnet unkalibriert)'] = judge_df         # immer vorhanden
_judge_versions['v2\n(Sonnet kalibriert)']   = judge_v2_df      # canonical nach Abschnitt 8

if 'judge_opus_df' in dir() and judge_opus_df is not None:
    _judge_versions['v3\n(Opus kalibriert)'] = judge_opus_df
else:
    print('⚠️  judge_opus_df fehlt — Abschnitt 9 ausführen.')

_oai_path = RESULTS_DIR / 'eval_llm_judge_openai.json'
if _oai_path.exists():
    _judge_versions[f'Cross-Vendor\n({OPENAI_JUDGE_MODEL})'] = judge_oai_df
else:
    print('⚠️  OpenAI-Judge-Datei fehlt — Abschnitt 11 ausführen.')

print(f'Judges geladen: {list(_judge_versions.keys())}')
print(f'Einträge je Judge: { {k: len(v) for k, v in _judge_versions.items()} }')

In [ ]:
# ── Krippendorff's α über alle geparten Judges ───────────────────────────────
judge_labels = list(_judge_versions.keys())
id_cols = ['pipeline_label', 'xai_model', 'instance_id']

# Gemeinsame Einheiten (Units) = Schnittmenge über alle Judges
_all_indices = [
    set(zip(df_[id_cols[0]], df_[id_cols[1]], df_[id_cols[2]]))
    for df_ in _judge_versions.values()
]
_common_units = sorted(_all_indices[0].intersection(*_all_indices[1:]))
print(f'Gemeinsame Units für α: {len(_common_units)} '
      f'(von max. {len(df)} — Überschneidung aller Judges)\n')

alpha_rows = []
for metric in ['faithfulness', 'clarity', 'completeness']:
    # reliability_data: (n_raters × n_units)
    reliability = []
    for df_ in _judge_versions.values():
        idx = df_.set_index(id_cols)[metric]
        row_scores = []
        for unit in _common_units:
            key = (unit[0], unit[1], unit[2])
            row_scores.append(float(idx.get(key, np.nan)))
        reliability.append(row_scores)

    alpha = krippendorff_alpha_interval(np.array(reliability))
    alpha_rows.append({'metric': metric.capitalize(), 'krippendorff_alpha': round(alpha, 3)})
    print(f'  Krippendorff α ({metric:14s}): {alpha:.3f}')

alpha_df = pd.DataFrame(alpha_rows).set_index('metric')
print()
print('Richtwerte: α < 0.20 = schwach, 0.20–0.60 = moderat, > 0.60 = gut (Krippendorff 2011)')

In [ ]:
# ── Paarweises Kendall's τ zwischen je zwei Judges ───────────────────────────
print('Paarweises Kendall\'s τ (Rangkorrelation auf Einzel-Score-Ebene, n=80 Units)')
print('τ > 0.6 = gute Übereinstimmung der Instanz-Rangfolge\n')

tau_tables = {}
for metric in ['faithfulness', 'clarity', 'completeness']:
    rows = []
    for i, (la, dfa) in enumerate(list(_judge_versions.items())):
        for lb, dfb in list(_judge_versions.items())[i+1:]:
            xa, xb = _align_scores(dfa, dfb, metric)
            if len(xa) < 3:
                rows.append({'Judge A': la.replace('\n', ' '),
                             'Judge B': lb.replace('\n', ' '),
                             'n': len(xa), 'tau': np.nan, 'p': np.nan})
                continue
            tau, pval = kendalltau(xa, xb)
            rows.append({'Judge A': la.replace('\n', ' '),
                         'Judge B': lb.replace('\n', ' '),
                         'n': len(xa),
                         'tau': round(tau, 3),
                         'p':   round(pval, 4)})
    tau_df = pd.DataFrame(rows)
    tau_tables[metric] = tau_df
    print(f'── {metric.upper()} ──')
    display(tau_df.to_string(index=False))
    print()

# ── Zusammenfassung und CSV-Export ───────────────────────────────────────────
agreement_summary = pd.concat(
    [df_.assign(metric=m) for m, df_ in tau_tables.items()],
    ignore_index=True,
)[['metric', 'Judge A', 'Judge B', 'n', 'tau', 'p']]
out_agr = RESULTS_DIR / 'eval_inter_judge_agreement.csv'
agreement_summary.to_csv(out_agr, index=False)

alpha_out = RESULTS_DIR / 'eval_krippendorff_alpha.csv'
alpha_df.to_csv(alpha_out)

print(f'Gespeichert: {out_agr}')
print(f'Gespeichert: {alpha_out}')
print()
print('=== ZUSAMMENFASSUNG ===')
print()
print('Krippendorff α über alle 4 Judges:')
display(alpha_df)
print()
print('Interpretation für das Paper:')
print('  – Hohe τ-Werte (> 0.6) zwischen allen Judge-Paaren → Ranking-Aussagen robust')
print('  – Niedriges α → Judges stimmen in absolutem Score weniger überein als in Rangfolge')
print('  – Kalibrierung (v2 vs v1) und Vendor-Wechsel (OAI vs Anthropic) beeinflussen')
print('    absolute Scores mehr als die relative Instanz-Rangfolge')